# Lesson 7: Training efficiency and model size

Measure parameter memory, accumulate gradients across small batches, and enable mixed precision when running on a CUDA GPU.

**How to run:** Select a Python kernel with PyTorch installed, then run each code cell from top to bottom with **Shift+Enter**. This notebook is self-contained; no other notebook needs to run first. Restart the kernel and run from the top to reset the experiment.

**Source:** This lesson was developed from the [reference conversation's roadmap](https://chatgpt.com/share/6aa56bca-4a1c-83e9-9153-1edcc7ff7e40). The reference supplies Lesson 1 and a topic outline; Lessons 2–12 are newly written implementations of those topics. Small examples demonstrate the mechanics; they are not trained assistants.


In [ ]:
import math
from pathlib import Path
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(42)
# Small tensors can be slower with many CPU threads.
torch.set_num_threads(1)
device = torch.device('cpu')
print('PyTorch:', torch.__version__, '| device:', device)


## Load and tokenize text

We reuse `input.txt`, resolving it from the notebook folder or repository root. `B` means batch size, `T` means context length, and `C` means vector width. Our file is only 80 characters, so the validation scores are noisy and text generation will be limited.


In [ ]:
input_path = Path('input.txt')
if not input_path.is_file():
    input_path = Path('chatgpt/input.txt')
text = input_path.read_text(encoding='utf-8')
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return ''.join(itos[int(i)] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.8 * len(data))
train_data, val_data = data[:split], data[split:]
block_size = min(8, len(train_data) - 1, len(val_data) - 1)
if block_size < 1:
    raise ValueError('input.txt needs enough text for train and validation sequences.')
batch_size = 4
print('Characters:', vocab_size, '| train:', len(train_data), '| validation:', len(val_data))
print('Context length:', block_size)


## Draw random batches

Choose starting positions within one split. Targets are the same window shifted right by one token. No window crosses from training into validation. The tokenizer vocabulary uses the full text so every validation character has an ID; model weights are updated only on training tokens.


In [ ]:
def get_batch(split_name='train'):
    if split_name not in ('train', 'val'):
        raise ValueError("Choose 'train' or 'val'.")
    source = train_data if split_name == 'train' else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in starts])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in starts])
    return x.to(device), y.to(device)

xb, yb = get_batch()
print('Input shape:', xb.shape, '| target shape:', yb.shape)
print('Input :', repr(decode(xb[0])))
print('Target:', repr(decode(yb[0])))
assert torch.equal(xb[:, 1:], yb[:, :-1])


## Causal multi-head attention

Each head compares queries to keys, then combines value vectors. A triangular mask prevents reading future tokens. This is the implementation developed in Lessons 3–4, included here so this notebook runs independently.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, width, heads, context_length, dropout=0.0):
        super().__init__()
        assert width % heads == 0
        self.heads = heads
        self.head_size = width // heads
        self.qkv = nn.Linear(width, 3 * width, bias=False)
        self.projection = nn.Linear(width, width)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('causal_mask', torch.tril(torch.ones(context_length, context_length, dtype=torch.bool)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # Give each head its own vector slice: [B, heads, T, head_size].
        q = q.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        k = k.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        v = v.reshape(B, T, self.heads, self.head_size).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_size)
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float('-inf'))
        weights = self.dropout(F.softmax(scores, dim=-1))
        out = (weights @ v).transpose(1, 2).contiguous().reshape(B, T, C)
        return self.projection(out)


## Transformer block

LayerNorm normalizes each token's features. Residual additions let information pass around attention and the MLP. The MLP expands each token vector, applies a nonlinear function, and projects it back.


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, width, heads, context_length, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(width)
        self.attention = CausalSelfAttention(width, heads, context_length, dropout)
        self.ln2 = nn.LayerNorm(width)
        self.mlp = nn.Sequential(
            nn.Linear(width, 4 * width), nn.GELU(),
            nn.Linear(4 * width, width), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


## Token and position embeddings → vocabulary scores

Token embeddings describe characters; learned position embeddings distinguish their positions. The final linear layer predicts the next character at every position. Cross entropy consumes raw scores (logits).


In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, context_length, width=32, heads=4, layers=2):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, width)
        self.position_embedding = nn.Embedding(context_length, width)
        self.blocks = nn.Sequential(*[
            TransformerBlock(width, heads, context_length) for _ in range(layers)
        ])
        self.final_norm = nn.LayerNorm(width)
        self.lm_head = nn.Linear(width, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.context_length:
            raise ValueError('Sequence exceeds context length.')
        positions = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        logits = self.lm_head(self.final_norm(self.blocks(x)))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

model = TinyGPT(vocab_size, block_size).to(device)
print('Parameters:', sum(p.numel() for p in model.parameters()))


## Choose a device and estimate parameter memory

CPU is the default elsewhere in this course. This lesson chooses CUDA if available, then Apple MPS, then CPU. The byte count below covers parameters only; training also needs gradients, optimizer state, activations, and temporary tensors. Dense attention scores grow quadratically with context length.


In [ ]:
import time
from contextlib import nullcontext

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
model = model.to(device)
parameter_count = sum(p.numel() for p in model.parameters())
parameter_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
print('Device:', device, '| parameters:', parameter_count, '| parameter MiB:', parameter_bytes / 2**20)

def gpt_parameter_count(vocab, context, width, heads, layers):
    # This formula matches TinyGPT, including biases and LayerNorm parameters.
    assert width % heads == 0
    return 2 * vocab * width + context * width + layers * (12 * width**2 + 10 * width) + 2 * width + vocab

assert gpt_parameter_count(vocab_size, block_size, 32, 4, 2) == parameter_count
print('Larger configuration parameter count:', gpt_parameter_count(5000, 256, 384, 6, 6))


## Accumulate gradients

Four micro-batches of size 4 produce an effective batch size of 16. Divide each loss by the number of accumulation steps and call the optimizer once after all four backward passes. This matches an averaged larger batch when model behavior is independent across examples, as it is here with dropout disabled.

On CUDA, autocast uses float16 for eligible operations, and gradient scaling reduces underflow risk. CPU and MPS use float32 in this example.


In [ ]:
accumulation_steps = 4
optimizer_steps = 20
use_amp = device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

def synchronize():
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()

model.train()
synchronize()
started = time.perf_counter()
for step in range(optimizer_steps):
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    for micro_step in range(accumulation_steps):
        x, y = get_batch()
        amp_context = torch.autocast('cuda', dtype=torch.float16) if use_amp else nullcontext()
        with amp_context:
            _, loss = model(x, y)
            scaled_loss = loss / accumulation_steps
        scaler.scale(scaled_loss).backward()
        total_loss += scaled_loss.item()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()
    if step % 5 == 0:
        print('Optimizer step:', step, '| mean micro-batch loss:', round(total_loss, 4))
synchronize()
elapsed = time.perf_counter() - started
tokens = optimizer_steps * accumulation_steps * batch_size * block_size
print('Effective batch:', batch_size * accumulation_steps)
print('Training tokens/second:', round(tokens / elapsed))


## Try it yourself

Compare accumulation steps 1 and 4 while keeping the number of training tokens equal. Estimate parameter memory for a 50-million-parameter float32 model. Use a larger, suitable corpus before allocating a large model; changing dimensions alone does not improve training data.
